In [9]:
# ==================================================
# 1. SETUP - WITH LOGGER PATH WORKAROUND
# ==================================================
import os
import sys
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import logging

# Set current directory
print(f"📁 Current directory: {os.getcwd()}")

# Add necessary paths
sys.path.append("..")
sys.path.append("../src")

# Set up logging manually to avoid the config issue
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-8s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)

plt.style.use('default')
sns.set_palette("husl")

📁 Current directory: /Users/sarayetel/Desktop/Research/Graph/RC-Element-Prediction-GNN/notebooks


In [10]:
# ==================================================
# 2. IMPORT WITH LOGGER PATCH
# ==================================================
print("📦 Importing modules with logger patch...")

# First, let's temporarily set an environment variable for the logger
import os
os.environ['LOGGER_CONFIG_PATH'] = "../configs/logger.yaml"
print(f"📝 Set LOGGER_CONFIG_PATH to: {os.environ['LOGGER_CONFIG_PATH']}")

# Now import
try:
    from src.data_manager.data_loader import HeteroDataLoader
    from src.data_manager.graph_builder import HeteroGraphBuilder
    print("✅ Modules imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\nTrying direct import...")
    
    import sys
    sys.path.insert(0, os.path.abspath(".."))
    
    from data_manager.data_loader import HeteroDataLoader
    from data_manager.graph_builder import HeteroGraphBuilder
    print("✅ Direct import successful!")

📦 Importing modules with logger patch...
📝 Set LOGGER_CONFIG_PATH to: ../configs/logger.yaml
✅ Modules imported successfully!


In [11]:
# ==================================================
# 3. LOAD CONFIG
# ==================================================
display(Markdown("### 📋 Loading Configuration"))

config_path = "../configs/base.yaml"
print(f"Loading config from: {config_path}")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Display key settings
config_summary = pd.DataFrame([
    {"Setting": "Project", "Value": config.get('project', {}).get('name', 'N/A')},
    {"Setting": "PE Enabled", "Value": config.get('data', {}).get('pe', {}).get('enabled', 'N/A')},
    {"Setting": "PE Type", "Value": config.get('data', {}).get('pe', {}).get('type', 'N/A')},
    {"Setting": "PE Dimension", "Value": config.get('data', {}).get('pe', {}).get('dim', 'N/A')},
    {"Setting": "Base Features", "Value": len(config.get('data', {}).get('feature_columns', []))}
])

display(config_summary)

### 📋 Loading Configuration

Loading config from: ../configs/base.yaml


,Setting,Value
0,Project,rc-element-prediction
1,PE Enabled,True
2,PE Type,hetero_laplacian
3,PE Dimension,8
4,Base Features,41


In [12]:
# ==================================================
# 4. INITIALIZE DATA LOADER AND LOAD DATA
# ==================================================
display(Markdown("### 📁 Loading Data"))

loader = HeteroDataLoader("../configs/base.yaml")
print("✅ DataLoader initialized")

# Load samples
train_data_list = loader.load_all_samples("train")
print(f"📊 Loaded {len(train_data_list)} samples")

# Show first few samples
sample_info = []
for i, sample in enumerate(train_data_list[:3]):
    beam_count = len(sample['nodes']['beam'])
    column_count = len(sample['nodes']['column'])
    edge_count = len(sample['edges_raw'])
    
    sample_info.append({
        "Sample": sample.get('sample_name', f'sample_{i}'),
        "Beams": beam_count,
        "Columns": column_count,
        "Total Nodes": beam_count + column_count,
        "Edges": edge_count
    })

sample_df = pd.DataFrame(sample_info)
display(sample_df)

# Show first sample details
first_sample = train_data_list[0]
display(Markdown(f"#### First Sample: {first_sample['sample_name']}"))
print(f"Beam columns: {len(first_sample['nodes']['beam'].columns)}")
print(f"Column columns: {len(first_sample['nodes']['column'].columns)}")
print(f"Feature columns (excluding Ele_Type, row_index): {len([col for col in first_sample['nodes']['beam'].columns if col not in ['Ele_Type', 'row_index']])}")

### 📁 Loading Data

22:14:51 | 📁 DATA   | ℹ️  INFO     | Initializing HeteroDataLoader with config: ../configs/base.yaml
22:14:51 | 📁 DATA   | ℹ️  INFO     | DataLoader initialized successfully
✅ DataLoader initialized
22:14:51 | 📁 DATA   | ℹ️  INFO     | Loading all samples from split: train
22:14:51 | 📁 DATA   | ℹ️  INFO     | Found 254 sample folders
22:14:51 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_1 (train)
22:14:52 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_1
22:14:52 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_10 (train)
22:14:52 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_10
22:14:52 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_100 (train)
22:14:52 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_100
22:14:52 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_101 (train)
22:14:52 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_101
22:14:52 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_102 (train)
22:14:52 |

,Sample,Beams,Columns,Total Nodes,Edges
0,sample_1,371,103,474,1740
1,sample_10,176,96,272,660
2,sample_100,205,109,314,600


#### First Sample: sample_1

Beam columns: 44
Column columns: 44
Feature columns (excluding Ele_Type, row_index): 42


In [13]:
# ==================================================
# 5. INITIALIZE GRAPH BUILDER WITH PATCH
# ==================================================
display(Markdown("### 🏗️ Initializing Graph Builder"))

# Create a patched GraphBuilder to avoid set_verbose issue
class PatchedGraphBuilder(HeteroGraphBuilder):
    """Patched version that fixes set_verbose method."""
    
    def set_verbose(self, verbose: bool):
        """Fixed version of set_verbose."""
        self.verbose = verbose
        level = logging.DEBUG if verbose else logging.INFO
        self.logger.setLevel(level)
        if verbose:
            self.logger.debug("Verbose mode enabled")

# Initialize
builder = PatchedGraphBuilder(config)
print("✅ GraphBuilder initialized (patched)")

# Show configuration
print(f"\n📊 Configuration:")
print(f"  PE enabled: {builder.pe_enabled}")
print(f"  PE type: {builder.pe_type}")
print(f"  PE dimension: {builder.pe_dim}")
print(f"  Base features: {len(config.get('data', {}).get('feature_columns', []))}")

### 🏗️ Initializing Graph Builder

22:15:23 | 🏗️ GRAPH  | ℹ️  INFO     | Initializing HeteroGraphBuilder
22:15:23 | 🏗️ GRAPH  | ℹ️  INFO     | GraphBuilder initialized successfully
✅ GraphBuilder initialized (patched)

📊 Configuration:
  PE enabled: True
  PE type: hetero_laplacian
  PE dimension: 8
  Base features: 41


In [14]:
# ==================================================
# 6. BUILD GRAPHS WITH PROPER ERROR HANDLING
# ==================================================
display(Markdown("### 🔨 Building Graphs"))

graphs = []
feature_counts = []

for i, sample in enumerate(train_data_list):
    sample_name = sample.get('sample_name', f'sample_{i}')
    print(f"\n[{i+1}/{len(train_data_list)}] Building: {sample_name}")
    
    # Build graph
    graph = builder.build_hetero_graph(sample)
    
    if graph is None:
        print("  ❌ Graph is None")
        continue
    
    graphs.append((sample_name, graph))
    
    try:
        # SAFELY check beam features
        if hasattr(graph, 'beam') and hasattr(graph.beam, 'x'):
            beam_x = graph.beam.x
            beam_shape = beam_x.shape
            print(f"  ✅ Beams: {beam_shape[0]} × {beam_shape[1]} features")
            
            # Calculate PE
            base_features = 42
            pe_features = max(0, beam_shape[1] - base_features)
            
            feature_counts.append({
                "sample": sample_name,
                "node_type": "beam",
                "nodes": beam_shape[0],
                "features": beam_shape[1],
                "base": base_features,
                "pe": pe_features,
                "has_pe": pe_features > 0
            })
        
        # SAFELY check column features
        if hasattr(graph, 'column') and hasattr(graph.column, 'x'):
            column_x = graph.column.x
            column_shape = column_x.shape
            print(f"  ✅ Columns: {column_shape[0]} × {column_shape[1]} features")
            
            # Calculate PE
            base_features = 42
            pe_features = max(0, column_shape[1] - base_features)
            
            feature_counts.append({
                "sample": sample_name,
                "node_type": "column",
                "nodes": column_shape[0],
                "features": column_shape[1],
                "base": base_features,
                "pe": pe_features,
                "has_pe": pe_features > 0
            })
        
        # SAFELY count edges - FIXED VERSION
        print("  📍 Checking edges...")
        edge_patterns = [
            ('beam', 'to', 'beam'),
            ('column', 'to', 'column'),
            ('beam', 'to', 'column'),
            ('column', 'to', 'beam')
        ]
        
        for edge_type in edge_patterns:
            # Convert tuple to string for attribute access
            edge_attr = f"{edge_type[0]}_{edge_type[1]}_{edge_type[2]}"
            
            # Check if this edge type exists in the graph
            if hasattr(graph, edge_attr):
                edge_data = getattr(graph, edge_attr)
                if hasattr(edge_data, 'edge_index'):
                    count = edge_data.edge_index.shape[1]
                    if count > 0:
                        print(f"    {edge_type}: {count} edges")
        
    except Exception as e:
        print(f"  ⚠️ Error analyzing graph: {e}")

### 🔨 Building Graphs


[1/254] Building: sample_1
22:15:23 | ⚙️ SYSTEM | ℹ️  INFO     | Adding hetero_laplacian positional encoding (dim=8)
22:15:23 | ⚙️ SYSTEM | ⚠️  WARNING  | Graph has 195 isolated nodes (52.6%)
22:15:23 | ⚙️ SYSTEM | ⚠️  WARNING  | Graph has 36 isolated nodes (35.0%)
22:15:24 | ⚙️ SYSTEM | ❌ ERROR    | Laplacian computation failed: module 'scipy.linalg' has no attribute 'ArpackNoConvergence'
22:15:24 | ⚙️ SYSTEM | ℹ️  INFO     | Computed PE for 2 node types
22:15:24 | ⚙️ SYSTEM | ℹ️  INFO     | Added PE to 2 node types
22:15:24 | 🏗️ GRAPH  | ℹ️  INFO     | Successfully built graph for sample_1
  📍 Checking edges...

[2/254] Building: sample_10
22:15:24 | ⚙️ SYSTEM | ℹ️  INFO     | Adding hetero_laplacian positional encoding (dim=8)
22:15:24 | ⚙️ SYSTEM | ⚠️  WARNING  | Graph has 104 isolated nodes (59.1%)
22:15:24 | ⚙️ SYSTEM | ⚠️  WARNING  | Graph has 70 isolated nodes (72.9%)
22:15:25 | ⚙️ SYSTEM | ❌ ERROR    | Laplacian computation failed: module 'scipy.linalg' has no attribute 'Arpa

In [15]:
# ==================================================
# 7. DISPLAY FEATURE ANALYSIS
# ==================================================
display(Markdown("### 📊 Feature Analysis Results"))

if feature_counts:
    # Create DataFrame
    feat_df = pd.DataFrame(feature_counts)
    
    # Display detailed table
    display(Markdown("#### 📋 Detailed Feature Counts"))
    display(feat_df[['sample', 'node_type', 'nodes', 'features', 'base_features', 'pe_features', 'has_pe']])
    
    # Summary statistics
    display(Markdown("#### 📈 Summary"))
    
    summary = feat_df.groupby('node_type').agg({
        'nodes': ['mean', 'min', 'max'],
        'features': ['mean', 'std', 'count'],
        'base_features': 'first',
        'pe_features': 'first',
        'has_pe': 'first'
    }).round(2)
    
    display(summary)
    
    # Check consistency
    beam_features = feat_df[feat_df['node_type'] == 'beam']['features'].unique()
    column_features = feat_df[feat_df['node_type'] == 'column']['features'].unique()
    
    print("\n🔍 Consistency Check:")
    if len(beam_features) == 1:
        print(f"✅ Beam features consistent: {beam_features[0]}")
    else:
        print(f"⚠️ Beam features vary: {beam_features}")
    
    if len(column_features) == 1:
        print(f"✅ Column features consistent: {column_features[0]}")
    else:
        print(f"⚠️ Column features vary: {column_features}")
    
    # Check if PE was added
    if feat_df['has_pe'].any():
        pe_dim = feat_df['pe_features'].iloc[0]
        total_features = feat_df['features'].iloc[0]
        base_features = feat_df['base_features'].iloc[0]
        
        print(f"\n🎯 Positional Encoding:")
        print(f"  Base features: {base_features}")
        print(f"  PE dimension: {pe_dim}")
        print(f"  Total features: {total_features}")
        print(f"  Formula: {base_features} + {pe_dim} = {total_features}")
        
        if pe_dim == config.get('data', {}).get('pe', {}).get('dim', 8):
            print(f"✅ PE dimension matches config: {pe_dim}")
        else:
            print(f"⚠️ PE dimension mismatch: expected {config.get('data', {}).get('pe', {}).get('dim', 8)}, got {pe_dim}")
    
    else:
        print(f"\n⚠️ No Positional Encoding detected")
        print(f"  Expected: 42 base + {config.get('data', {}).get('pe', {}).get('dim', 8)} PE = {42 + config.get('data', {}).get('pe', {}).get('dim', 8)} total")
        print(f"  Actual: {feat_df['features'].iloc[0]} features")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Feature composition
    for i, (sample, node_type) in enumerate(zip(feat_df['sample'], feat_df['node_type'])):
        row = feat_df[(feat_df['sample'] == sample) & (feat_df['node_type'] == node_type)].iloc[0]
        
        if i < 3:  # Show first 3
            labels = [f"Base ({int(row['base_features'])})"]
            sizes = [row['base_features']]
            
            if row['pe_features'] > 0:
                labels.append(f"PE ({int(row['pe_features'])})")
                sizes.append(row['pe_features'])
            
            axes[i].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
            axes[i].set_title(f"{sample}\n{node_type}")
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ No feature data collected")

### 📊 Feature Analysis Results

❌ No feature data collected


In [16]:
# ==================================================
# 8. FIX THE LOGGER CONFIG ISSUE
# ==================================================
display(Markdown("### 🔧 Permanent Fix for Logger Config"))

print("To permanently fix the logger config issue, update your logger.py file:")
print("\nCurrent problematic line (probably in src/utils/logger.py):")
print('```python')
print('def __init__(self, config_path: str = "configs/logger.yaml"):')
print('```')

print("\nChange it to one of these options:")
print("\nOption 1: Use absolute path:")
print('```python')
print(f'def __init__(self, config_path: str = "{os.path.abspath("../configs/logger.yaml")}"):')
print('```')

print("\nOption 2: Use relative path:")
print('```python')
print('def __init__(self, config_path: str = "../configs/logger.yaml"):')
print('```')

print("\nOption 3: Smart detection (recommended):")
print('```python')
print('def __init__(self, config_path: str = None):')
print('    if config_path is None:')
print('        # Try multiple locations')
print('        possible_paths = [')
print('            "configs/logger.yaml",')
print('            "../configs/logger.yaml",')
print('            "../../configs/logger.yaml",')
print('        ]')
print('        ')
print('        for path in possible_paths:')
print('            if os.path.exists(path):')
print('                config_path = path')
print('                break')
print('        else:')
print('            config_path = "configs/logger.yaml"  # Default')
print('```')

print("\n📋 Quick fix command (run in terminal):")
print(f'cd {os.path.dirname(os.getcwd())}  # Go to project root')
print("sed -i '' \"s/config_path: str = \\\"configs\\/logger.yaml\\\"/config_path: str = \\\"..\\/configs\\/logger.yaml\\\"/\" src/utils/logger.py")

### 🔧 Permanent Fix for Logger Config

To permanently fix the logger config issue, update your logger.py file:

Current problematic line (probably in src/utils/logger.py):
```python
def __init__(self, config_path: str = "configs/logger.yaml"):
```

Change it to one of these options:

Option 1: Use absolute path:
```python
def __init__(self, config_path: str = "/Users/sarayetel/Desktop/Research/Graph/RC-Element-Prediction-GNN/configs/logger.yaml"):
```

Option 2: Use relative path:
```python
def __init__(self, config_path: str = "../configs/logger.yaml"):
```

Option 3: Smart detection (recommended):
```python
def __init__(self, config_path: str = None):
    if config_path is None:
        # Try multiple locations
        possible_paths = [
            "configs/logger.yaml",
            "../configs/logger.yaml",
            "../../configs/logger.yaml",
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                config_path = path
                break
        else:
            

### New Debug

In [17]:
# ==================================================
# MINIMAL NOTEBOOK TO SHOW FEATURE COUNTS
# ==================================================
import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Setup
sys.path.append("..")
sys.path.append("../src")

# Import
from src.data_manager.data_loader import HeteroDataLoader
from src.data_manager.graph_builder import HeteroGraphBuilder

# Load config and data
import yaml
with open("../configs/base.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Initialize
loader = HeteroDataLoader("../configs/base.yaml")
builder = HeteroGraphBuilder(config)

# Load just ONE sample
print("📊 Loading one sample...")
train_data_list = loader.load_all_samples("train")[:1]  # Just one!

if train_data_list:
    sample = train_data_list[0]
    sample_name = sample['sample_name']
    print(f"📋 Sample: {sample_name}")
    
    # Build graph
    print(f"🏗️ Building graph...")
    graph = builder.build_hetero_graph(sample)
    
    if graph is not None:
        print(f"✅ Graph built successfully!")
        
        # ==================================================
        # SIMPLE, GUARANTEED WAY TO CHECK FEATURES
        # ==================================================
        
        print("\n🔍 Checking graph structure...")
        
        # Method 1: Direct attribute check
        print("\n📋 Method 1: Direct attribute access")
        
        # Check if 'beam' exists
        if hasattr(graph, 'beam'):
            print("✅ Found 'beam' in graph")
            beam_obj = graph.beam
            
            # Check if beam has 'x' (features)
            if hasattr(beam_obj, 'x'):
                beam_x = beam_obj.x
                print(f"   Beam features shape: {beam_x.shape}")
                print(f"   → {beam_x.shape[0]} beams × {beam_x.shape[1]} features")
                
                # Calculate PE
                if beam_x.shape[1] > 42:
                    pe_dim = beam_x.shape[1] - 42
                    print(f"   → Composition: 42 base + {pe_dim} PE = {beam_x.shape[1]} total")
                else:
                    print(f"   → {beam_x.shape[1]} features (no PE)")
            else:
                print("   ❌ Beam has no 'x' attribute")
        else:
            print("❌ No 'beam' attribute in graph")
        
        # Check if 'column' exists
        if hasattr(graph, 'column'):
            print("✅ Found 'column' in graph")
            column_obj = graph.column
            
            if hasattr(column_obj, 'x'):
                column_x = column_obj.x
                print(f"   Column features shape: {column_x.shape}")
                print(f"   → {column_x.shape[0]} columns × {column_x.shape[1]} features")
                
                # Calculate PE
                if column_x.shape[1] > 42:
                    pe_dim = column_x.shape[1] - 42
                    print(f"   → Composition: 42 base + {pe_dim} PE = {column_x.shape[1]} total")
                else:
                    print(f"   → {column_x.shape[1]} features (no PE)")
            else:
                print("   ❌ Column has no 'x' attribute")
        else:
            print("❌ No 'column' attribute in graph")
        
        # ==================================================
        # Method 2: Check all attributes
        # ==================================================
        print("\n📋 Method 2: All graph attributes")
        print("Attributes in graph object:")
        for attr in dir(graph):
            if not attr.startswith('_') and not callable(getattr(graph, attr)):
                attr_value = getattr(graph, attr)
                print(f"  {attr}: {type(attr_value)}")
        
        # ==================================================
        # Method 3: Check using node_types
        # ==================================================
        print("\n📋 Method 3: Using node_types")
        if hasattr(graph, 'node_types'):
            print(f"Node types: {list(graph.node_types)}")
            
            for node_type in graph.node_types:
                try:
                    node_data = graph[node_type]
                    if hasattr(node_data, 'x'):
                        shape = node_data.x.shape
                        print(f"  {node_type}: {shape[0]} × {shape[1]}")
                except:
                    print(f"  {node_type}: Could not access")
        else:
            print("No 'node_types' attribute")
        
        # ==================================================
        # Method 4: Check metadata
        # ==================================================
        print("\n📋 Method 4: Metadata")
        
        # Check if graph has any metadata
        metadata_attrs = ['sample_name', 'match_percentage', 'connectivity_ratio']
        for attr in metadata_attrs:
            if hasattr(graph, attr):
                value = getattr(graph, attr)
                print(f"  {attr}: {value}")
        
        # ==================================================
        # SUMMARY
        # ==================================================
        print("\n" + "="*60)
        print("🎯 SUMMARY")
        print("="*60)
        
        # Get feature counts if available
        beam_features = None
        column_features = None
        
        if hasattr(graph, 'beam') and hasattr(graph.beam, 'x'):
            beam_features = graph.beam.x.shape[1]
            
        if hasattr(graph, 'column') and hasattr(graph.column, 'x'):
            column_features = graph.column.x.shape[1]
        
        if beam_features and column_features:
            print(f"✅ FEATURE COUNTS FOUND:")
            print(f"   Beam: {beam_features} features")
            print(f"   Column: {column_features} features")
            
            # Check PE
            if beam_features > 42:
                pe_dim = beam_features - 42
                print(f"\n🎯 POSITIONAL ENCODING:")
                print(f"   Base features: 42")
                print(f"   PE dimension: {pe_dim}")
                print(f"   Total: {beam_features} = 42 + {pe_dim}")
                
                if pe_dim == config.get('data', {}).get('pe', {}).get('dim', 8):
                    print(f"   ✅ PE matches config ({pe_dim})")
                else:
                    print(f"   ⚠️ PE mismatch: config says {config.get('data', {}).get('pe', {}).get('dim', 8)}, got {pe_dim}")
            else:
                print(f"\n⚠️ NO PE DETECTED")
                print(f"   Expected: 42 + {config.get('data', {}).get('pe', {}).get('dim', 8)} = {42 + config.get('data', {}).get('pe', {}).get('dim', 8)}")
                print(f"   Actual: {beam_features}")
        else:
            print("❌ Could not determine feature counts")
        
    else:
        print("❌ Graph is None")
else:
    print("❌ No samples loaded")

22:18:29 | 📁 DATA   | ℹ️  INFO     | Initializing HeteroDataLoader with config: ../configs/base.yaml
22:18:29 | 📁 DATA   | ℹ️  INFO     | DataLoader initialized successfully
22:18:29 | 🏗️ GRAPH  | ℹ️  INFO     | Initializing HeteroGraphBuilder
22:18:29 | 🏗️ GRAPH  | ℹ️  INFO     | GraphBuilder initialized successfully
📊 Loading one sample...
22:18:29 | 📁 DATA   | ℹ️  INFO     | Loading all samples from split: train
22:18:29 | 📁 DATA   | ℹ️  INFO     | Found 254 sample folders
22:18:29 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_1 (train)
22:18:29 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_1
22:18:29 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_10 (train)
22:18:30 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_10
22:18:30 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_100 (train)
22:18:30 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_100
22:18:30 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_101 (train)
22:18:30 | 📁

In [18]:
# ==================================================
# ULTRA-SIMPLE DEBUG SCRIPT
# ==================================================
import os
import sys
sys.path.append("..")
sys.path.append("../src")

from src.data_manager.data_loader import HeteroDataLoader
from src.data_manager.graph_builder import HeteroGraphBuilder
import yaml

# Load config
with open("../configs/base.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Initialize
loader = HeteroDataLoader("../configs/base.yaml")
builder = HeteroGraphBuilder(config)

# Load one sample
samples = loader.load_all_samples("train")[:1]
sample = samples[0]

# Build graph
graph = builder.build_hetero_graph(sample)

# SIMPLE DEBUG - JUST PRINT EVERYTHING
print("="*60)
print("DEBUG: GRAPH OBJECT INSPECTION")
print("="*60)

# 1. Print type
print(f"Type: {type(graph)}")

# 2. Print all attributes
print("\nAll attributes (non-callable):")
for attr_name in dir(graph):
    if not attr_name.startswith('_'):
        try:
            attr_value = getattr(graph, attr_name)
            if not callable(attr_value):
                print(f"  {attr_name}: {type(attr_value)}")
        except:
            print(f"  {attr_name}: [ERROR ACCESSING]")

# 3. Try to access beam
print("\nTrying to access beam...")
try:
    beam_data = graph.beam
    print(f"  graph.beam: {type(beam_data)}")
    
    if hasattr(beam_data, 'x'):
        beam_x = beam_data.x
        print(f"  graph.beam.x shape: {beam_x.shape}")
        print(f"  → Features: {beam_x.shape[1]}")
    else:
        print("  ❌ beam has no 'x'")
        
except Exception as e:
    print(f"  ❌ Error: {e}")

# 4. Try to access column
print("\nTrying to access column...")
try:
    column_data = graph.column
    print(f"  graph.column: {type(column_data)}")
    
    if hasattr(column_data, 'x'):
        column_x = column_data.x
        print(f"  graph.column.x shape: {column_x.shape}")
        print(f"  → Features: {column_x.shape[1]}")
    else:
        print("  ❌ column has no 'x'")
        
except Exception as e:
    print(f"  ❌ Error: {e}")

# 5. Try dictionary access
print("\nTrying dictionary access...")
try:
    # PyG HeteroData can be accessed like a dict
    if 'beam' in graph:
        beam_x = graph['beam'].x
        print(f"  graph['beam'].x shape: {beam_x.shape}")
    
    if 'column' in graph:
        column_x = graph['column'].x
        print(f"  graph['column'].x shape: {column_x.shape}")
        
except Exception as e:
    print(f"  ❌ Dictionary access error: {e}")

print("\n" + "="*60)
print("If you see feature shapes above, that's your answer!")
print("="*60)

22:19:10 | 📁 DATA   | ℹ️  INFO     | Initializing HeteroDataLoader with config: ../configs/base.yaml
22:19:10 | 📁 DATA   | ℹ️  INFO     | DataLoader initialized successfully
22:19:10 | 🏗️ GRAPH  | ℹ️  INFO     | Initializing HeteroGraphBuilder
22:19:10 | 🏗️ GRAPH  | ℹ️  INFO     | GraphBuilder initialized successfully
22:19:10 | 📁 DATA   | ℹ️  INFO     | Loading all samples from split: train
22:19:10 | 📁 DATA   | ℹ️  INFO     | Found 254 sample folders
22:19:10 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_1 (train)
22:19:11 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_1
22:19:11 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_10 (train)
22:19:11 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_10
22:19:11 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_100 (train)
22:19:11 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_100
22:19:11 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_101 (train)
22:19:11 | 📁 DATA   | ℹ️  INFO     |